# Phase 2 — Baseline RT-DETR Training (Colab T4)

## Overview
This notebook fine-tunes RT-DETR-L (COCO pretrained) on the 4-class humanoid robot parts dataset and records baseline mAP before any corruption-robustness training.

## Pre-requisites
1. **Phase 1 complete** — dataset annotated, VIZ 1.A/B/C saved.
2. **Dataset uploaded to Google Drive** at:
   `My Drive/robot-perception/data/annotated/`
   The folder must contain `images/{train,val,test,calibration}/` and `labels/{train,val,test,calibration}/`.
3. Runtime set to **T4 GPU** (Runtime → Change runtime type → T4 GPU).

## What this notebook does
| Step | Action |
|------|--------|
| Setup | Mount Drive, install ultralytics, define paths |
| Data  | Copy dataset to local `/content/` for fast I/O |
| YAML  | Write local data.yaml (4 classes) |
| Train | RT-DETR-L, 100 epochs, patience=20, batch=16 |
| VIZ 2.A | Training curve (loss + val mAP) |
| VIZ 2.B | Confusion matrix on val set |
| VIZ 2.C | 10 qualitative detections on test set |
| Check | Print mAP@0.5 / mAP@0.5:0.95 vs targets, PASS/FAIL |
| Save  | Copy best.pt to Drive |

## Target metrics
| Metric | Target |
|--------|--------|
| mAP@0.5 | ≥ 0.70 |
| mAP@0.5:0.95 | ≥ 0.50 |

**If mAP@0.5 < 0.60 after training, do NOT proceed to Phase 3.** Diagnose annotation errors and class balance first.

In [ ]:
# ── Cell 2: Setup — Mount Drive, install ultralytics, define paths ──────────────
from google.colab import drive
drive.mount('/content/drive')

# Install / upgrade ultralytics (includes RT-DETR support)
import subprocess
subprocess.run(['pip', 'install', '-q', '--upgrade', 'ultralytics'], check=True)

import os, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import torch

# ── Path constants (edit PROJECT_DIR only if your Drive layout differs) ─────────
PROJECT_DIR  = '/content/drive/MyDrive/robot-perception'
DATA_DIR     = f'{PROJECT_DIR}/data/annotated'
MODEL_DIR    = f'{PROJECT_DIR}/models/baseline'
RESULTS_DIR  = f'{PROJECT_DIR}/results'
FIG_DIR      = f'{RESULTS_DIR}/figures'

for d in [MODEL_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = ['arm', 'leg', 'torso', 'head']   # 4 classes, YOLO IDs 0-3

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('PROJECT_DIR:', PROJECT_DIR)

## Step 1 — Copy dataset to local disk

Drive I/O is slow during training (~5-10× slower than local SSD). Copy the full annotated dataset to `/content/robot_data/` once so all epoch reads are fast.

In [ ]:
# ── Cell 3: Copy dataset from Drive to local /content/ ──────────────────────────
LOCAL_DATA = '/content/robot_data'

if os.path.exists(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
shutil.copytree(DATA_DIR, LOCAL_DATA)
print(f'Copied {DATA_DIR} → {LOCAL_DATA}')

# Verify split counts
for split in ['train', 'val', 'test', 'calibration']:
    img_dir = f'{LOCAL_DATA}/images/{split}'
    if os.path.exists(img_dir):
        n = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        print(f'  {split}: {n} images')
    else:
        print(f'  {split}: directory not found at {img_dir}')

## Step 2 — Write local data.yaml

Ultralytics expects a YAML file with `path`, `train`, `val`, `test`, `nc`, and `names`. We write a fresh one pointing at the local copy so training never touches Drive.

In [ ]:
# ── Cell 4: Write local data.yaml ───────────────────────────────────────────────
YAML_PATH = '/content/robot_parts.yaml'

yaml_content = f"""path: {LOCAL_DATA}
train: images/train
val:   images/val
test:  images/test

nc: 4
names: ['arm', 'leg', 'torso', 'head']
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print('Wrote', YAML_PATH)
print(yaml_content)

## Step 3 — Train RT-DETR-L

`rtdetr-l.pt` downloads COCO pretrained weights automatically (~115 MB) on first run.

Key hyperparameters:
- `epochs=100`, `patience=20` — early stopping on val mAP
- `batch=16` — safe for T4 (15 GB VRAM) at 640×640; lower to 8 if OOM
- `lr0=1e-4` — conservative LR for fine-tuning from COCO pretrained
- Results saved locally to `/content/runs/baseline/`; best weights copied to Drive after training

In [ ]:
# ── Cell 5: Train RT-DETR-L ─────────────────────────────────────────────────────
from ultralytics import RTDETR

model = RTDETR('rtdetr-l.pt')   # downloads COCO pretrained weights on first run

train_results = model.train(
    data=YAML_PATH,
    epochs=100,
    patience=20,        # early stopping — stops if val mAP doesn't improve for 20 epochs
    imgsz=640,
    batch=16,           # reduce to 8 if you get CUDA out-of-memory
    lr0=1e-4,
    weight_decay=1e-4,
    warmup_epochs=3,
    device=0,           # GPU
    project='/content/runs',
    name='baseline',
    exist_ok=True,
    save=True,
)

BEST_LOCAL = '/content/runs/baseline/weights/best.pt'
print('Training complete.')
print('Best weights at:', BEST_LOCAL)

## VIZ 2.A — Training curve

Plot box loss (train + val) and val mAP@0.5 vs epoch from `results.csv`. Saved to Drive as `viz2a_training_curve.png`.

In [ ]:
# ── Cell 6: VIZ 2.A — Training curve ────────────────────────────────────────────
results_csv = '/content/runs/baseline/results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()   # ultralytics sometimes adds leading spaces

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('VIZ 2.A — RT-DETR-L Baseline Training', fontsize=14)

# Left: box loss
ax = axes[0]
if 'train/box_loss' in df.columns:
    ax.plot(df['epoch'], df['train/box_loss'], label='train box loss', color='steelblue')
if 'val/box_loss' in df.columns:
    ax.plot(df['epoch'], df['val/box_loss'], label='val box loss', color='orange')
ax.set_xlabel('Epoch')
ax.set_ylabel('Box Loss')
ax.set_title('Box Loss')
ax.legend()
ax.grid(alpha=0.3)

# Right: val mAP@0.5
ax = axes[1]
map_col = None
for candidate in ['metrics/mAP50(B)', 'metrics/mAP_0.5']:
    if candidate in df.columns:
        map_col = candidate
        break

if map_col:
    ax.plot(df['epoch'], df[map_col], label='val mAP@0.5', color='green')
    ax.axhline(y=0.70, color='red', linestyle='--', linewidth=1.5, label='target 0.70')
    ax.set_ylim(0, 1.05)
else:
    ax.text(0.5, 0.5, 'mAP column not found\nCheck df.columns', ha='center', va='center',
            transform=ax.transAxes, fontsize=11, color='red')

ax.set_xlabel('Epoch')
ax.set_ylabel('mAP@0.5')
ax.set_title('Validation mAP@0.5')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
out_path = f'{FIG_DIR}/viz2a_training_curve.png'
plt.savefig(out_path, dpi=150)
plt.show()
print(f'Saved: {out_path}')

## VIZ 2.B — Confusion matrix on val set

Run `model.val()` on the val split to get per-class predictions, then plot a normalised confusion matrix with seaborn. Saved to Drive as `viz2b_confusion_matrix.png`.

In [ ]:
# ── Cell 7: VIZ 2.B — Confusion matrix ──────────────────────────────────────────
import glob

# Load best weights
model = RTDETR(BEST_LOCAL)

# Run val to generate confusion matrix artefacts
val_metrics = model.val(data=YAML_PATH, split='val')

# Ultralytics saves a confusion_matrix.png in the run dir — use it if available
cm_png_candidates = glob.glob('/content/runs/baseline/confusion_matrix*.png')
if cm_png_candidates:
    # Copy pre-rendered one from ultralytics
    src = sorted(cm_png_candidates)[-1]
    out_path = f'{FIG_DIR}/viz2b_confusion_matrix.png'
    shutil.copy2(src, out_path)
    from IPython.display import Image, display
    display(Image(src))
    print(f'Saved (ultralytics render): {out_path}')
else:
    # Fallback: build from the ConfusionMatrix object
    cm_obj = val_metrics.confusion_matrix
    if cm_obj is not None:
        matrix = cm_obj.matrix   # shape: (nc+1, nc+1) including background
        # Trim background row/col if present, keep nc×nc
        nc = len(CLASS_NAMES)
        cm = matrix[:nc, :nc].astype(float)
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_norm = np.divide(cm, row_sums, where=row_sums != 0)

        fig, ax = plt.subplots(figsize=(7, 6))
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    vmin=0, vmax=1, ax=ax)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title('VIZ 2.B — Confusion Matrix (val, normalised)')
        plt.tight_layout()
        out_path = f'{FIG_DIR}/viz2b_confusion_matrix.png'
        plt.savefig(out_path, dpi=150)
        plt.show()
        print(f'Saved: {out_path}')
    else:
        print('No confusion matrix available. Rerun model.val() with save_json=True.')

## VIZ 2.C — Qualitative detections on 10 test images

Run inference on 10 randomly sampled test images. Overlay ground truth boxes (green) and predicted boxes (blue) with class labels and confidence scores. Saved to Drive as `viz2c_qualitative_detections.png`.

In [ ]:
# ── Cell 8: VIZ 2.C — Qualitative detections on 10 test images ──────────────────
test_img_dir = f'{LOCAL_DATA}/images/test'
test_imgs = sorted([
    os.path.join(test_img_dir, f)
    for f in os.listdir(test_img_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])
sample = random.sample(test_imgs, min(10, len(test_imgs)))

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
fig.suptitle('VIZ 2.C — GREEN: ground truth | BLUE: predictions', fontsize=13)

for ax, img_path in zip(axes.flatten(), sample):
    result = model(img_path, verbose=False)[0]
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    h, w = img.shape[:2]

    # Draw predictions (blue)
    for box, cls_id, conf in zip(
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.cls.cpu().numpy().astype(int),
        result.boxes.conf.cpu().numpy()
    ):
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), (80, 80, 255), 2)
        label = f'{CLASS_NAMES[cls_id]} {conf:.2f}'
        cv2.putText(img, label, (x1, max(y1 - 6, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (80, 80, 255), 2)

    # Draw ground truth (green)
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = f'{LOCAL_DATA}/labels/test/{stem}.txt'
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (80, 200, 80), 2)
                cv2.putText(img, CLASS_NAMES[cls_id], (x1, max(y1 - 6, 0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (80, 200, 80), 1)

    ax.imshow(img)
    ax.axis('off')
    ax.set_title(os.path.basename(img_path)[:22], fontsize=8)

# Legend patches
pred_patch = mpatches.Patch(color=(80/255, 80/255, 255/255), label='Prediction')
gt_patch   = mpatches.Patch(color=(80/255, 200/255, 80/255), label='Ground truth')
fig.legend(handles=[pred_patch, gt_patch], loc='lower center', ncol=2,
           fontsize=11, bbox_to_anchor=(0.5, 0.01))

plt.tight_layout(rect=[0, 0.04, 1, 1])
out_path = f'{FIG_DIR}/viz2c_qualitative_detections.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## Step 4 — Metrics check (PASS / FAIL)

Evaluate on the held-out **test** split and check both mAP thresholds. Print `PASS` or `FAIL` for each. Do not proceed to Phase 3 if mAP@0.5 < 0.60.

In [ ]:
# ── Cell 9: Metrics check — test set ────────────────────────────────────────────
test_metrics = model.val(data=YAML_PATH, split='test')

map50   = float(test_metrics.box.map50)
map5095 = float(test_metrics.box.map)

TARGET_MAP50   = 0.70
TARGET_MAP5095 = 0.50

print('=' * 45)
print('  Phase 2 — Metrics Check (test split)')
print('=' * 45)
print(f'  mAP@0.5       : {map50:.4f}   target >= {TARGET_MAP50}  '
      f'{"PASS" if map50 >= TARGET_MAP50 else "FAIL"}')
print(f'  mAP@0.5:0.95  : {map5095:.4f}   target >= {TARGET_MAP5095}  '
      f'{"PASS" if map5095 >= TARGET_MAP5095 else "FAIL"}')
print('=' * 45)

# Per-class breakdown
p_vals = test_metrics.box.p   # array, one per class
r_vals = test_metrics.box.r
print()
print(f'  {"Class":<10}  {"Precision":>10}  {"Recall":>8}')
print(f'  {"-"*10}  {"-"*10}  {"-"*8}')
for name, p, r in zip(CLASS_NAMES, p_vals, r_vals):
    print(f'  {name:<10}  {p:>10.4f}  {r:>8.4f}')

print()
if map50 < 0.60:
    print('FAIL  mAP@0.5 < 0.60 — DO NOT proceed to Phase 3.')
    print('      Diagnose: check annotation quality, class balance,')
    print('      and consider training for more epochs.')
elif map50 >= TARGET_MAP50 and map5095 >= TARGET_MAP5095:
    print('Phase 2 COMPLETE')
else:
    print('PARTIAL — one or more targets not met.')
    print('Consider: more epochs, data augmentation, or annotation review.')

## Step 5 — Save best weights to Drive

Copy `best.pt` from the local run directory to `models/baseline/best.pt` on Drive so it persists after the Colab session ends.

In [ ]:
# ── Cell 10: Save best.pt to Drive ──────────────────────────────────────────────
BEST_DRIVE = f'{MODEL_DIR}/best.pt'
os.makedirs(MODEL_DIR, exist_ok=True)
shutil.copy2(BEST_LOCAL, BEST_DRIVE)
print(f'Saved best.pt to Drive: {BEST_DRIVE}')

# Sanity check — print file sizes
local_mb = os.path.getsize(BEST_LOCAL) / 1e6
drive_mb = os.path.getsize(BEST_DRIVE) / 1e6
print(f'  local:  {local_mb:.1f} MB')
print(f'  Drive:  {drive_mb:.1f} MB')

## Phase 2 Completion Checklist

Before opening `03_corruption_benchmark.ipynb`, confirm every item below:

**Training**
- [ ] RT-DETR-L trained for ≥ 20 epochs without early stopping (or stopped because val mAP plateaued)
- [ ] `best.pt` saved to `models/baseline/best.pt` on Drive

**Metrics (test split)**
- [ ] mAP@0.5 ≥ 0.70
- [ ] mAP@0.5:0.95 ≥ 0.50

**Visualisations (all saved to `results/figures/` on Drive)**
- [ ] `viz2a_training_curve.png` — loss and mAP curves
- [ ] `viz2b_confusion_matrix.png` — normalised confusion matrix
- [ ] `viz2c_qualitative_detections.png` — 10 test images with GT + predictions

**Gate**
- If mAP@0.5 < 0.60: **stop here**, diagnose before proceeding
- If mAP@0.5 ≥ 0.70 and mAP@0.5:0.95 ≥ 0.50 and all VIZs saved: open `03_corruption_benchmark.ipynb`